# 🤖 Fase eliminatoria - Mundial 2026: Análisis de Octavos de Final

## 1. Inicialización del Entorno y Carga de Datos
En esta sección importamos las librerías analíticas fundamentales (`pandas`, `numpy`, `scipy.stats`) y cargamos la **Matriz de Características Consolidada** (`features_rendimiento_2026.csv`). Esta matriz contiene las métricas históricas de rendimiento y las variables de forma de cada selección clasificada.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import random

# 1. Cargar matriz de características con ruta dinámica absoluta
ruta_features = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'processed', 'features_rendimiento_2026.csv'))

# Verificación de existencia del archivo base
if os.path.exists(ruta_features):
    df_features = pd.read_csv(ruta_features)
    # Homologación de columnas para el pipeline analítico
    columnas_mapeo = {
        'Equipo': 'equipo',
        'Efectividad_Historica': 'efectividad_historica_%',
        'Puntos_Por_Partido': 'puntos_por_pj_2026',
        'Expectativa_Goles_Poisson': 'expectativa_goles_poisson'
    }
    df_features = df_features.rename(columns=columnas_mapeo)
    print(f"✅ Matriz de características cargada con éxito. Total de selecciones: {len(df_features)}")
else:
    print(f"❌ No se encontró el archivo base en la ruta: {ruta_features}")

## 2. Definición del Cuadro del Torneo y Cruce de Variables
Estructuramos las 8 llaves oficiales de los Octavos de Final del Mundial. Mediante operaciones de unión (*merge*), cruzamos los datos de rendimiento de los equipos locales (`home_team`) y visitantes (`away_team`) para calcular los primeros deltas teóricos de rendimiento.

In [ ]:
# 1. Definición del cuadro real de Octavos de Final
octavos_proyectados = [
    {"home_team": "Paraguay", "away_team": "France"},
    {"home_team": "Canada", "away_team": "Morocco"},
    {"home_team": "Brazil", "away_team": "Norway"},
    {"home_team": "Mexico", "away_team": "England"},
    {"home_team": "Portugal", "away_team": "Spain"},
    {"home_team": "United States", "away_team": "Belgium"},
    {"home_team": "Argentina", "away_team": "Egypt"},
    {"home_team": "Switzerland", "away_team": "Colombia"}
]

df_r16 = pd.DataFrame(octavos_proyectados)

# Cruzar datos base de rendimiento para cálculos rápidos de deltas preliminares
df_h = df_features[['equipo', 'puntos_por_pj_2026', 'efectividad_historica_%']].rename(columns=lambda x: x + '_home' if x != 'equipo' else 'home_team')
df_a = df_features[['equipo', 'puntos_por_pj_2026', 'efectividad_historica_%']].rename(columns=lambda x: x + '_away' if x != 'equipo' else 'away_team')

df_octavos_crossed = df_r16.merge(df_h, on='home_team').merge(df_a, on='away_team')
df_octavos_crossed['diff_forma_2026'] = df_octavos_crossed['puntos_por_pj_2026_home'] - df_octavos_crossed['puntos_por_pj_2026_away']

print("📋 Estructura de llaves de Octavos de Final cargada correctamente.")

## 3. Análisis Exploratorio: Gradiente de Forma Reciente
Antes de correr las simulaciones matemáticas, desplegamos un gráfico divergente utilizando `Seaborn`. Este mapa visual nos muestra qué equipos llegan con mayor inercia ganadora en el torneo actual evaluando el diferencial de puntos obtenidos por partido.

In [ ]:
# 1. Preparación estética y ordenamiento predictivo de los datos
plt.figure(figsize=(13, 7))
sns.set_theme(style="whitegrid")

df_octavos_crossed['llave'] = df_octavos_crossed['home_team'] + " vs " + df_octavos_crossed['away_team']
df_grafico = df_octavos_crossed.sort_values(by='diff_forma_2026', ascending=False)

# 2. Despliegue del gráfico divergente con Seaborn utilizando el valor numérico para mapear colores
sns.barplot(
    data=df_grafico, 
    x='diff_forma_2026', 
    y='llave', 
    palette='coolwarm',
    hue='diff_forma_2026',
    legend=False
)

plt.axvline(x=0, color='black', linestyle='--', linewidth=1.2)
plt.title('Divergencia Teórica de Forma Reciente (Puntos por Partido en 2026)', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Delta de Forma (Local - Visitante)', fontsize=11)
plt.ylabel('Cruce de Octavos', fontsize=11)
plt.tight_layout()
plt.show()

## 4. Modelado Predictivo: Distribución de Poisson y Simulaciones de Monte Carlo
Esta sección aloja el núcleo del algoritmo predictivo:
1. **Fuerzas Combinadas:** Balanceamos un 60% el momento actual en el mundial y un 40% el peso de la historia para ajustar las expectativas de gol ($\lambda$).
2. **Distribución de Poisson:** Evaluamos probabilísticamente los escenarios de prórroga y penaltis.
3. **Generador Estocástico Unitario:** Proyecta un marcador exacto con minutos de goles ponderados para el segundo tiempo.
4. **Simulación de Monte Carlo:** Ejecuta **1,000 torneos alternativos en paralelo** utilizando vectores de NumPy para obtener la probabilidad real de supervivencia/clasificación de cada país.

In [ ]:
import scipy.stats as stats
import random
import numpy as np

# ==============================================================================
# 5. MODELADO PREDICTIVO: PROBABILIDADES DE ALARGUE, PENALTIS Y MARCADOR EXACTO
# ==============================================================================

GOLES_PROMEDIO_TORNEO = 2.6

def calcular_y_simular_encuentro(equipo_home, equipo_away):
    dict_stats = df_features.set_index('equipo').to_dict('index')
    
    if equipo_home not in dict_stats or equipo_away not in dict_stats:
        return None
        
    feat_h, feat_a = dict_stats[equipo_home], dict_stats[equipo_away]
    
    # Modelo matemático de fuerzas combinadas (60% Momento / 40% Historia)
    fuerza_h = (feat_h['puntos_por_pj_2026'] * 0.6) + ((feat_h['efectividad_historica_%'] / 25) * 0.4)
    fuerza_a = (feat_a['puntos_por_pj_2026'] * 0.6) + ((feat_a['efectividad_historica_%'] / 25) * 0.4)
    
    lambda_h = max(0.4, min((fuerza_h / fuerza_a if fuerza_a > 0 else 1) * (GOLES_PROMEDIO_TORNEO / 2), 2.5))
    lambda_a = max(0.4, min((fuerza_a / fuerza_h if fuerza_h > 0 else 1) * (GOLES_PROMEDIO_TORNEO / 2), 2.5))
    
    # 📊 CÁLCULO VECTORIAL DE PROBABILIDADES (Matriz de Poisson hasta 8 goles)
    max_goles = 9
    prob_goles_h = [stats.poisson.pmf(g, lambda_h) for g in range(max_goles)]
    prob_goles_a = [stats.poisson.pmf(g, lambda_a) for g in range(max_goles)]
    
    # Probabilidad de Tiempo Extra: Sumatoria de P(G_h == G_a)
    prob_tiempo_extra = sum(prob_goles_h[g] * prob_goles_a[g] for g in range(max_goles))
    
    # Probabilidad de Penaltis: Probabilidad de ir a tiempo extra * Factor de permanencia (fatiga)
    lambda_te_h = lambda_h * 0.33 * 0.85
    lambda_te_a = lambda_a * 0.33 * 0.85
    prob_empate_te = sum(stats.poisson.pmf(g, lambda_te_h) * stats.poisson.pmf(g, lambda_te_a) for g in range(5))
    prob_penaltis = prob_tiempo_extra * prob_empate_te

    # Simular cantidad de goles en 90' por Poisson
    goles_h, goles_a = stats.poisson.rvs(lambda_h), stats.poisson.rvs(lambda_a)
    
    # Generador cronológico de minutos
    def generar_minutos(cantidad):
        bloques = list(range(1, 91))
        pesos = [1 if m <= 45 else 1.6 for m in bloques]
        return sorted([f"{m}'" for m in random.choices(bloques, weights=pesos, k=cantidad)], key=lambda x: int(x.replace("'", "")))

    min_h = generar_minutos(goles_h)
    min_a = generar_minutos(goles_a)

    # Resolución de clasificación en caso de empate
    ganador_penales = None
    if goles_h == goles_a:
        prob_home_win = fuerza_h / (fuerza_h + fuerza_a)
        ganador_penales = random.choices([equipo_home, equipo_away], weights=[prob_home_win, 1 - prob_home_win])[0]

    return {
        "goles_h": goles_h, "goles_a": goles_a,
        "min_h": min_h, "min_a": min_a,
        "prob_te": prob_tiempo_extra * 100,
        "prob_pen": prob_penaltis * 100,
        "ganador_penales": ganador_penales,
        "lambda_h": lambda_h, "lambda_a": lambda_a,
        "fuerza_h": fuerza_h, "fuerza_a": fuerza_a
    }

# ==============================================================================
# MOTOR DE SIMULACIÓN DE MONTE CARLO (1,000 ITERACIONES VECTORIZADAS)
# ==============================================================================
def simular_monte_carlo(equipo_home, equipo_away, n_simulaciones=1000):
    base_stats = calcular_y_simular_encuentro(equipo_home, equipo_away)
    if not base_stats: return None
    
    # Generación masiva vectorizada en memoria
    goles_sim_h = stats.poisson.rvs(base_stats["lambda_h"], size=n_simulaciones)
    goles_sim_a = stats.poisson.rvs(base_stats["lambda_a"], size=n_simulaciones)
    
    # Conteo de clasificaciones directas
    clasifica_home = (goles_sim_h > goles_sim_a).sum()
    clasifica_away = (goles_sim_a > goles_sim_h).sum()
    empates = (goles_sim_h == goles_sim_a).sum()
    
    # Desempate de penaltis mediante distribución binomial masiva
    prob_desempate_home = base_stats["fuerza_h"] / (base_stats["fuerza_h"] + base_stats["fuerza_a"])
    desempates_home = np.random.binomial(empates, prob_desempate_home)
    
    total_home_win = clasifica_home + desempates_home
    total_away_win = clasifica_away + (empates - desempates_home)
    
    return {
        "pct_home": (total_home_win / n_simulaciones) * 100,
        "pct_away": (total_away_win / n_simulaciones) * 100
    }

# --- RUNTIME GLOBAL DE ANÁLISIS ---
print("====== 🏆 PROYECCIÓN ESTADÍSTICA AVANZADA Y MONTE CARLO (OCTAVOS) ======\n")
res_montecarlo_global = {}

for _, partido in df_r16.iterrows():
    home, away = partido['home_team'], partido['away_team']
    res = calcular_y_simular_encuentro(home, away)
    mc = simular_monte_carlo(home, away, n_simulaciones=1000)
    
    if res and mc:
        res_montecarlo_global[f"{home} vs {away}"] = mc
        print(f"🏟️  {home} vs {away}")
        print(f"    📊 Probabilidad de Tiempo Extra: {res['prob_te']:.2f}%")
        print(f"    🎯 Probabilidad de llegar a Penaltis: {res['prob_pen']:.2f}%")
        print(f"    🎲 Simulación Monte Carlo (1,000 ejecuciones):")
        print(f"        ➡️ Probabilidad Clasificación {home}: {mc['pct_home']:.1f}%")
        print(f"        ➡️ Probabilidad Clasificación {away}: {mc['pct_away']:.1f}%")
        print(f"    🎴 Marcador Único (90'): {home} {res['goles_h']} - {res['goles_a']} {away}")
        print("-" * 70)

## 5. Pipeline de Exportación de Datos: Formato Homologado para Tableau
Finalmente, recuperamos los porcentajes obtenidos a partir del motor estadístico de Monte Carlo para la llave específica de **Brasil vs. Noruega**. Formateamos los números estructurando las variables con comas decimales (`,`) y exportamos el archivo usando como separador el punto y coma (`;`). Esto garantiza una compatibilidad absoluta con las configuraciones regionales de Tableau evitando cualquier error en tus dashboards.

In [ ]:
import pandas as pd
import os

# 1. Recuperar los datos del diccionario y las probabilidades de Monte Carlo calculadas
pct_brasil = res_montecarlo_global["Brazil vs Norway"]["pct_home"]
pct_noruega = res_montecarlo_global["Brazil vs Norway"]["pct_away"]

# 2. Estructura simétrica en 2 filas limpias (idéntica a la de Canadá)
match_data = {
    'Equipo': ['Brazil', 'Norway'],
    'Probabilidad_Clasificacion': [pct_brasil, pct_noruega],
    'Efectividad_Historica': [68.40, 45.10],
    'Puntos_Por_Partido': [2.66, 1.85],
    'Expectativa_Goles_Poisson': [1.91, 0.88]
}

df_match = pd.DataFrame(match_data)

# 3. Formatear decimales numéricos con coma (,) para evitar errores regionales en Tableau
columnas_num = ['Probabilidad_Clasificacion', 'Efectividad_Historica', 'Puntos_Por_Partido', 'Expectativa_Goles_Poisson']
for col in columnas_num:
    df_match[col] = df_match[col].round(2).astype(str).str.replace('.', ',', regex=False)

# 4. Establecer ruta absoluta de guardado en /data/processed/
ruta_output = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'processed', 'Reporte_Brasil_Noruega.csv'))

# 5. Exportar usando punto y coma (;) como delimitador estándar para tu Tableau
os.makedirs(os.path.dirname(ruta_output), exist_ok=True)
df_match.to_csv(ruta_output, index=False, encoding='utf-8-sig', sep=';')

print(f"✅ ¡Archivo dinámico de Monte Carlo exportado con éxito para Tableau!")
print(f"📍 Destino: {ruta_output}")